In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from dotenv import load_dotenv

In [2]:
load_dotenv()
model1 = ChatOpenAI(model='gpt-4.1', temperature = 0.1)

In [3]:
parser = JsonOutputParser()

In [4]:
template1 = PromptTemplate(
    template='I have a interview for {designation} position, generate the notes for it. \n {format_instruction}',
    input_variables=['designation'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

template2 = PromptTemplate(
    template='I have a interview for {designation} position \n so give me the 15 top questions and answer on it. \n {format_instruction}',
    input_variables=['designation'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

In [5]:
from langchain_core.runnables import RunnableParallel

In [6]:
parallel_chain = RunnableParallel({
    'notes': template1 | model1 | parser,
    'qa': template2 | model1 | parser 
})

In [7]:
template3 = PromptTemplate(
    template='Merge the generated {notes} and {qa} \n {format_instruction}',
    input_variables=['notes','qa'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

In [8]:
merge_chain = template3 | model1 | parser

In [9]:
chain = parallel_chain | merge_chain

In [10]:
result = chain.invoke({'designation': 'Technical Research Analyst'})

In [11]:
result

{'Position': 'Technical Research Analyst',
 'Overview': 'A Technical Research Analyst is responsible for analyzing technical data, market trends, and industry developments to provide actionable insights and recommendations. The role often involves working with large datasets, creating reports, and supporting decision-making processes.',
 'Key Responsibilities': ['Collect and analyze technical and market data from various sources.',
  'Monitor industry trends, emerging technologies, and competitor activities.',
  'Prepare detailed research reports, presentations, and dashboards.',
  'Collaborate with cross-functional teams (engineering, product, business).',
  'Develop and maintain databases and data visualization tools.',
  'Present findings and recommendations to stakeholders.',
  'Support strategic planning and business development initiatives.'],
 'Required Skills': ['Strong analytical and problem-solving skills.',
  'Proficiency in data analysis tools (Excel, SQL, Python, R, etc.).

In [12]:
chain.get_graph().print_ascii()

              +-------------------------+              
              | Parallel<notes,qa>Input |              
              +-------------------------+              
                  **               **                  
               ***                   ***               
             **                         **             
 +----------------+                +----------------+  
 | PromptTemplate |                | PromptTemplate |  
 +----------------+                +----------------+  
          *                                 *          
          *                                 *          
          *                                 *          
   +------------+                    +------------+    
   | ChatOpenAI |                    | ChatOpenAI |    
   +------------+                    +------------+    
          *                                 *          
          *                                 *          
          *                                 *   

In [13]:
result['Sample Interview Questions']

['Describe your experience with data analysis and visualization tools.',
 'How do you stay updated with the latest industry trends and technologies?',
 'Can you explain a complex technical concept to a non-technical audience?',
 'Describe a time when your research influenced a business decision.',
 'How do you ensure the accuracy and reliability of your research?',
 'What steps do you take to prioritize multiple research projects?']

In [14]:
result['Questions_and_Answers']

[{'question': '1. What is the role of a Technical Research Analyst?',
  'answer': 'A Technical Research Analyst is responsible for analyzing data, trends, and patterns in technology or financial markets to provide actionable insights. They use various tools and methodologies to collect, process, and interpret data, and present their findings to support decision-making.'},
 {'question': '2. What tools and software are you proficient in for data analysis?',
  'answer': 'I am proficient in tools such as Microsoft Excel, SQL, Python (with libraries like Pandas and NumPy), R, Tableau, and Power BI for data analysis and visualization.'},
 {'question': '3. How do you approach a new research project?',
  'answer': 'I start by understanding the objectives and requirements, then gather relevant data, clean and preprocess it, perform exploratory data analysis, apply appropriate analytical methods, and finally interpret and present the results.'},
 {'question': '4. Can you explain the difference b